# SIG13 GEP Null Scoring

Score the existing sPCA components against a **true-null benchmark**: cells carrying only
non-targeting linker guide barcodes (the same 9 barcodes characterized in
`analysis/interaction_scoring_null/01_glmGamPoi_single_term_null_slurm.r`), pseudobulked by
individual barcode identity instead of by real ligand. This mirrors
`07_spca/02_spca_waggr_scoring.ipynb` exactly through the scaling step (same population,
same scale reference) and only diverges at the pseudobulk-aggregation step, so
the null waggr scores are on identical footing to the real ones.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import scanpy as sc
import pandas as pd
import decoupler as dc
import matplotlib as mpl
import matplotlib.pyplot as plt

# Repo-relative paths (inputs: imports_stable/, outputs: analysis_outs/)
from pathlib import Path
_here = Path.cwd().resolve()
REPO_DIR = str(next(p for p in [_here, *_here.parents] if (p / "imports_stable").is_dir()))
IMPORTS_DIR = f"{REPO_DIR}/imports_stable"
OUTS_DIR = f"{REPO_DIR}/analysis_outs/02_combinatorial_screen_signalseq_SIG13"
import sys
sys.path.insert(0, f"{REPO_DIR}/functions")
import scanpy_custom as scc

In [2]:
rna = sc.read_h5ad(f"{IMPORTS_DIR}/SIG13/scanpy_outs/SIG13_doublets_DSB7.h5ad")

In [3]:
spca_components = pd.read_csv(f"{IMPORTS_DIR}/SIG13/analysis_outs/spca/zscore_degs_allLigands_0.1_alpha1.0_sPCA_loadings.csv")
net = spca_components.rename(columns={'gene':'target',
                                      'spca_component':'source',
                                      'loading':'weight'})

# define genes used in scoring (same components as the real analysis - no refitting)
comp_genes = net['target'].unique().tolist()

In [4]:
# subset and scale - identical to 07_spca/02_spca_waggr_scoring.ipynb, so the null stays on the
# same scaling reference (computed across the full 349,678-cell object) as the real scores
rna_sub = rna[:,comp_genes].copy()
rna_sub.X = rna_sub.layers['log1p_norm']
sc.pp.scale(rna_sub)

/data1/rudenska/EYW/mamba_envs/scanpy_standard2/lib/python3.14/functools.py:982: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


## Build the true-null population

Recovers the 9 individual non-targeting linker guide barcodes from
`feature_call_DSB7` (collapsed to a single `"linker"` label in
`ligand_call_DSB7`), restricted to singlet double-linker cells - identical
population definition to `analysis/interaction_scoring_null/01_glmGamPoi_single_term_null_slurm.r`.

Each barcode ID has its underscore stripped (e.g. `linker1_round1` ->
`linker1round1`) so that the downstream R annotation notebook's
`separate(interaction, into = c("ligand1","ligand2","replicate"), sep = "_")`
parsing - which assumes ligand tokens contain no underscores, true for real
ligand names like `IL2`/`CCL1` - works unchanged on the null pseudo-ligand
tokens too.

In [5]:
round1_linkers = ["linker1_round1", "linker2_round1", "linker3_round1"]
round2_linkers = ["linker4_round2", "linker5_round2", "linker6_round2",
                  "linker7_round2", "linker8_round2", "linker9_round2"]

def get_linker_feats(feature_call, linker_set):
    feats = str(feature_call).split('|')
    return [f for f in feats if f in linker_set]

obs = rna_sub.obs.copy()
# cast away from pandas Categorical dtype before .apply(): a list-returning function
# applied to a Categorical Series routes through Categorical.map(), which tries to
# hash the (list-valued) mapped results and fails with "unhashable type: list"
feature_call_str = obs['feature_call_DSB7'].astype(str)
obs['round1_linker_feats'] = feature_call_str.apply(get_linker_feats, linker_set=round1_linkers)
obs['round2_linker_feats'] = feature_call_str.apply(get_linker_feats, linker_set=round2_linkers)

null_mask = (
    (obs['ligand_call_round1_DSB7'].astype(str) == 'linker') &
    (obs['ligand_call_round2_DSB7'].astype(str) == 'linker') &
    (obs['round1_linker_feats'].apply(len) == 1) &
    (obs['round2_linker_feats'].apply(len) == 1)
)

null_obs = obs.loc[null_mask].copy()
null_obs['round1_linker_id'] = null_obs['round1_linker_feats'].apply(lambda x: x[0])
null_obs['round2_linker_id'] = null_obs['round2_linker_feats'].apply(lambda x: x[0])

# underscore-free tokens (see markdown above)
null_obs['round1_token'] = null_obs['round1_linker_id'].str.replace('_', '', regex=False)
null_obs['round2_token'] = null_obs['round2_linker_id'].str.replace('_', '', regex=False)
null_obs['pseudo_ligand_call'] = null_obs['round1_token'] + '_' + null_obs['round2_token']

print(f"True-null population size: {null_obs.shape[0]} cells")

True-null population size: 12257 cells


In [6]:
# QC: cell counts per pseudo-condition x replicate (mirrors the crosstabs already
# produced for the GLM null - sparse combos should be visible here before modeling)
qc_counts = null_obs.groupby(['pseudo_ligand_call', 'replicate'], observed=True).size().unstack(fill_value=0)
qc_counts['total'] = qc_counts.sum(axis=1)
qc_counts.sort_values('total', ascending=False)

replicate,rep1,rep2,total
pseudo_ligand_call,,,
linker3round1_linker8round2,0,1450,1450
linker3round1_linker9round2,0,1417,1417
linker2round1_linker8round2,1238,0,1238
linker2round1_linker9round2,1205,0,1205
linker3round1_linker7round2,0,1144,1144
linker2round1_linker7round2,1115,0,1115
linker2round1_linker6round2,857,0,857
linker3round1_linker6round2,0,840,840
linker1round1_linker4round2,321,284,605


In [7]:
rna_null = rna_sub[null_obs.index].copy()
rna_null.obs['pseudo_ligand_call'] = null_obs['pseudo_ligand_call'].values
rna_null.obs['replicate'] = null_obs['replicate'].values

# calculate pseudobulk over the null population, grouped by pseudo-condition instead
# of ligand_call_DSB7
rna_null_pb = sc.get.aggregate(rna_null, by=['pseudo_ligand_call', 'replicate'], func='mean')
rna_null_pb.obs_names.tolist()[:5]

['linker1round1_linker4round2_rep1',
 'linker1round1_linker4round2_rep2',
 'linker1round1_linker5round2_rep1',
 'linker1round1_linker5round2_rep2',
 'linker2round1_linker4round2_rep1']

In [8]:
dc.mt.waggr(rna_null_pb, net, tmin=5, verbose=True, layer='mean', times=0)

2026-07-14 14:01:03 | [INFO] waggr - Running waggr


2026-07-14 14:01:03 | [INFO] Extracted omics mat with 20 rows (observations) and 6610 columns (features)


2026-07-14 14:01:03 | [INFO] Network adjacency matrix has 6610 unique features and 78 unique sources


2026-07-14 14:01:03 | [INFO] waggr - using function _wmean


2026-07-14 14:01:03 | [INFO] waggr - using _wmean for the first time, will need to be compiled


2026-07-14 14:01:03 | [INFO] waggr - calculating scores for 78 sources across 20 observations


2026-07-14 14:01:04 | [INFO] waggr - adjusting p-values by FDR


2026-07-14 14:01:04 | [INFO] waggr - done


In [9]:
# Extract score_waggr from rna_null_pb.obsm
null_score_df = pd.DataFrame(rna_null_pb.obsm['score_waggr'],
                              index=rna_null_pb.obs_names,
                              columns=rna_null_pb.obsm['score_waggr'].columns)
null_score_df = null_score_df.rename_axis('interaction')

# sanity check: every row name should split into exactly 3 parts on "_"
# (pseudo_ligand1, pseudo_ligand2, replicate), matching the real annotation
# notebook's separate(..., sep = "_") parsing
bad_names = [name for name in null_score_df.index if name.count('_') != 2]
assert not bad_names, f"unexpected underscore count in pseudobulk row names: {bad_names}"
print(f"{null_score_df.shape[0]} pseudobulk rows, {null_score_df.shape[1]} components - all row names split cleanly.")

20 pseudobulk rows, 78 components - all row names split cleanly.


In [10]:
out_dir = f"{OUTS_DIR}/spca/spca_null"
os.makedirs(out_dir, exist_ok=True)
null_score_df.to_csv(os.path.join(out_dir, "zscore_degs_null_0.1_alpha1.0_waggr_score.csv"))